<a href="https://colab.research.google.com/github/DangLeUyen/Reinforcement-Learning/blob/main/U1_Huggy_fetch_stick.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Objectives

- Understand **the state space, action space and reward function used to train Huggy**.
- **Train the Huggy** to fetch the stick.
- Be able to play **with your trained Huggy directly in the browser**.

** Step by Step**
1. Clone the repo
2. Setup the Virtual Environment
3. Install the dependencies
4. Download and move the environment zip file



#### 1. Clone the repo

In [20]:
%%capture
# Clone the repository (can take 3min)
!git clone --depth 1 https://github.com/Unity-Technologies/ml-agents

#### 2. Setup the environment

In [2]:
# Colab's Current Python Version (Incompatible with ML-Agents)
!python --version

Python 3.12.13


In [21]:
# Install virtualenv and create a virtual environment
!pip install virtualenv
!virtualenv myenv

# !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
# !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Download and install Miniconda
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local

# Activate Miniconda and install Python ver 3.10.12
!source /usr/local/bin/activate
!conda install -q -y --prefix /usr/local python=3.10.12 ujson  # Specify the version here

# Set environment variables for Python and conda paths
!export PYTHONPATH=/usr/local/lib/python3.10/site-packages/
!export CONDA_PREFIX=/usr/local/envs/myenv

created virtual environment CPython3.13.13.final.0-64-x86_64 in 181ms
  creator CPython3Posix(dest=/content/ml-agents/ml-agents/myenv, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.1.2
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator,XonshActivator
--2026-06-12 00:01:41--  https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.191.158, 104.16.32.241, 2606:4700::6810:20f1, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.191.158|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 163179296 (156M) [application/octet-stream]
Saving to: ‘Miniconda3-latest-Linux-x86_64.sh’

Miniconda3-latest-L 100%[===================>] 155.62M   230MB/s    in 0.7s    

2026-06-12 00:01:42 (230 MB/s) - ‘Minicon

In [12]:
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r


accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [22]:
# Python Version in New Virtual Environment (Compatible with ML-Agents)
!python --version

Python 3.10.12


#### 3. Install the dependencies

In [33]:
%%capture
# Go inside the repository and install the package (can take 3min)
%cd ml-agents
!/usr/local/bin/python -m pip install -e ./ml-agents-envs
!/usr/local/bin/python -m pip install -e ./ml-agents

#### 4. Download and move the environment zip file in `./trained-envs-executables/linux/`


In [24]:
!mkdir ./trained-envs-executables
!mkdir ./trained-envs-executables/linux

In [25]:
!wget "https://github.com/huggingface/Huggy/raw/main/Huggy.zip" -O ./trained-envs-executables/linux/Huggy.zip

--2026-06-12 00:03:45--  https://github.com/huggingface/Huggy/raw/main/Huggy.zip
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://media.githubusercontent.com/media/huggingface/Huggy/main/Huggy.zip [following]
--2026-06-12 00:03:46--  https://media.githubusercontent.com/media/huggingface/Huggy/main/Huggy.zip
Resolving media.githubusercontent.com (media.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to media.githubusercontent.com (media.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 39214997 (37M) [application/zip]
Saving to: ‘./trained-envs-executables/linux/Huggy.zip’

./trained-envs-exec 100%[===================>]  37.40M  --.-KB/s    in 0.1s    

2026-06-12 00:03:46 (269 MB/s) - ‘./trained-envs-executables/linux/Huggy.zip’ saved [39214997

In [26]:
%%capture
!unzip -d ./trained-envs-executables/linux/ ./trained-envs-executables/linux/Huggy.zip

In [27]:
# Make sure the file is accessible
!chmod -R 755 ./trained-envs-executables/linux/Huggy

## How the environment works

### The State Space: what Huggy "perceives."

Huggy doesn't "see" his environment. Instead, we provide him information about the environment:

- The target (stick) position
- The relative position between himself and the target
- The orientation of his legs.

Given all this information, Huggy **can decide which action to take next to fulfill his goal**.

### The Action Space: what moves Huggy can do
**Joint motors drive huggy legs**. It means that to get the target, Huggy needs to **learn to rotate the joint motors of each of his legs correctly so he can move**.

### The Reward Function

The reward function is designed so that **Huggy will fulfill his goal** : fetch the stick.

Remember that one of the foundations of Reinforcement Learning is the *reward hypothesis*: a goal can be described as the **maximization of the expected cumulative reward**.

Here, our goal is that Huggy **goes towards the stick but without spinning too much**. Hence, our reward function must translate this goal.

Our reward function:
- *Orientation bonus*: we **reward him for getting close to the target**.
- *Time penalty*: a fixed-time penalty given at every action to **force him to get to the stick as fast as possible**.
- *Rotation penalty*: we penalize Huggy if **he spins too much and turns too quickly**.
- *Getting to the target reward*: we reward Huggy for **reaching the target**.

#### Create a huggy config file

Create Huggy.yaml in `/content/ml-agents/config/ppo`

In [31]:
%%writefile config/ppo/Huggy.yaml
behaviors:
  Huggy:
    trainer_type: ppo
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: linear
    network_settings:
      normalize: true
      hidden_units: 512
      num_layers: 3
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.995
        strength: 1.0
    checkpoint_interval: 200000
    keep_checkpoints: 15
    max_steps: 2e6
    time_horizon: 1000
    summary_freq: 50000

Overwriting config/ppo/Huggy.yaml


#### Train the agent

**launch mlagents-learn and select the executable containing the environment.**


1. `mlagents-learn <config>`: the path where the hyperparameter config file is.
2. `--env`: where the environment executable is.
3. `--run-id`: the name you want to give to your training run id.
4. `--no-graphics`: to not launch the visualization during the training.

Train the model and use the `--resume` flag to continue training in case of interruption.

> It will fail first time when you use `--resume`, try running the block again to bypass the error.

In [54]:
# # Install the missing 'tensorboard' package
# !/usr/local/bin/python -m pip install tensorboard

# # Install the missing 'onnxscript' package
# !/usr/local/bin/python -m pip install onnxscript

# # Downgrade protobuf to a compatible version (e.g., 3.20.3)
# !/usr/local/bin/python -m pip install protobuf==3.20.3

# Re-running the training command with --force to overwrite existing data
!/usr/local/bin/python -m mlagents.trainers.learn /content/ml-agents/config/ppo/Huggy.yaml --env=/content/ml-agents/trained-envs-executables/linux/Huggy/Huggy --run-id="Huggy2" --force --no-graphics

  Using cached protobuf-7.35.1-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
Using cached protobuf-7.35.1-cp310-abi3-manylinux2014_x86_64.whl (327 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlagents-envs 1.2.0.dev0 requires protobuf<3.21,>=3.6, but you have protobuf 7.35.1 which is incompatible.
  Using cached protobuf-3.20.3-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl.metadata (679 bytes)
Using cached protobuf-3.20.3-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl (1.1 MB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.35.1
    Uninstalling protobuf-7.35.1:
      Successfully uninstalled protobuf-7.35.1
ERROR: pip's dependenc